In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score
import random

/Users/kschille/micromamba/envs/info-decom/lib/python3.12/site-packages/scikits/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# --- 1. Generate Dummy Data (for demonstration) ---
# Replace with your actual data
N = 100  # Number of trials
nneurons = 50  # Total number of neurons
ns = 10  # Number of neurons to sample for training

# Dummy trial array (binary classification)
trial_array = np.random.randint(0, 2, N)

# Dummy neural data (N trials x nneurons)
neural_data_array = np.random.rand(N, nneurons) * 10

print(f"Trial array shape: {trial_array.shape}")
print(f"Neural data array shape: {neural_data_array.shape}")

Trial array shape: (100,)
Neural data array shape: (100, 50)


In [4]:
# --- 2. Define the Decoding Function ---
def decode_with_sampled_neurons(trial_data, neural_data, ns_neurons, n_iterations=10):
    """
    Trains and evaluates a logistic regression model by sampling disjoint sets of neurons.

    Args:
        trial_data (np.array): 1D array of trial labels (N,).
        neural_data (np.array): 2D array of neural activity (N, nneurons).
        ns_neurons (int): Number of neurons to use for training in each iteration.
        n_iterations (int): Number of times to sample disjoint neuron sets and evaluate.
        test_size (float): Proportion of the data to use as test set for train/test split.

    Returns:
        list: A list of accuracy scores from each iteration.
    """
    n_total_neurons = neural_data.shape[1]
    if ns_neurons > n_total_neurons:
        raise ValueError("ns_neurons cannot be greater than the total number of neurons.")
    if ns_neurons * 2 > n_total_neurons and n_iterations > 1:
        print(
            "Warning: With ns_neurons * 2 > n_total_neurons, it might be difficult to find truly disjoint sets for multiple iterations."
        )

    micomputationarray = []

    for _ in range(n_iterations):
        # 1. Randomly sample 'ns_neurons' for training
        available_neurons = list(range(n_total_neurons))
        training_neuron_indices = random.sample(available_neurons, ns_neurons)

        # 2. Identify remaining neurons
        remaining_neurons = [n for n in available_neurons if n not in training_neuron_indices]

        # 3. Sample 'ns_neurons' from the remaining (disjoint set) for testing
        if len(remaining_neurons) < ns_neurons:
            print(
                f"Warning: Not enough disjoint neurons left for a test set of size {ns_neurons}. Returning empty"
            )
            return np.asarray([])
        else:
            testing_neuron_indices = random.sample(remaining_neurons, ns_neurons)

        y = trial_data

        X_train_decoder_subset = neural_data[:, training_neuron_indices]
        X_test_decoder_subset = neural_data[
            :, testing_neuron_indices
        ]  # Use disjoint neurons for evaluation

        model = LogisticRegression(solver="liblinear", random_state=42, max_iter=1000)
        model.fit(X_train_decoder_subset, y)  # Fit on all trials

        y_pred = model.predict(X_test_decoder_subset)
        micomputationarray.append([y, y_pred])

    return np.asarray(micomputationarray)

In [5]:
def decode_with_sampled_neurons_pid(trial_data, neural_data, ns_neurons, n_iterations=10):
    """
    Trains and evaluates a logistic regression model by sampling disjoint sets of neurons.

    Args:
        trial_data (np.array): 1D array of trial labels (N,).
        neural_data (np.array): 2D array of neural activity (N, nneurons).
        ns_neurons (int): Number of neurons to use for training in each iteration.
        n_iterations (int): Number of times to sample disjoint neuron sets and evaluate.
        test_size (float): Proportion of the data to use as test set for train/test split.

    Returns:
        list: A list of accuracy scores from each iteration.
    """
    n_total_neurons = neural_data.shape[1]
    if ns_neurons > n_total_neurons:
        raise ValueError("ns_neurons cannot be greater than the total number of neurons.")
    if ns_neurons * 2 > n_total_neurons and n_iterations > 1:
        print(
            "Warning: With ns_neurons * 2 > n_total_neurons, it might be difficult to find truly disjoint sets for multiple iterations."
        )

    pid_computation_array = []

    for _ in range(n_iterations):
        # not doing cv here; probably not ideal but whatever; iterations should take care of this
        available_neurons = list(range(n_total_neurons))
        training_neurons = random.sample(available_neurons, ns_neurons)

        remaining_neurons = [n for n in available_neurons if n not in training_neurons]
        if len(remaining_neurons) < 2 * ns_neurons:
            print("This is not going to work, not enough disjoint neurons")
            return np.asarray([])

        neuron_set_a = random.sample(remaining_neurons, ns_neurons)
        used_neurons = training_neurons + neuron_set_a

        remaining_neurons = [n for n in available_neurons if n not in used_neurons]
        neuron_set_b = random.sample(remaining_neurons, ns_neurons)

        y = trial_data

        X_train_set = neural_data[:, training_neurons]
        X_set_a = neural_data[:, neuron_set_a]
        X_set_b = neural_data[:, neuron_set_b]

        model = LogisticRegression(solver="liblinear", random_state=42, max_iter=1000)
        model.fit(X_train_set, y)  # Fit on all trials

        y_pred_a = model.predict(X_set_a)
        y_pred_b = model.predict(X_set_b)

        pid_computation_array.append([y, y_pred_a, y_pred_b])

    return np.asarray(pid_computation_array)

In [24]:
results_mi = decode_with_sampled_neurons(
    trial_data=trial_array, neural_data=neural_data_array, ns_neurons=5
)

In [ ]:
results_pid = decode_with_sampled_neurons_pid(
    trial_data=trial_array, neural_data=neural_data_array, ns_neurons=5
)

In [68]:
results_pid.shape

(10, 3, 100)

In [6]:
from ibl_info.measures.information_measures import (
    mi_unbiased,
    pid_unbiased,
    correct_trivariate_mi,
    entropy,
    compute_probability_distribution,
)
from sklearn.metrics import mutual_info_score

In [ ]:
# # need to do more repeats; definitely. This gamut of tests run properly.

# np.sum(
#     pid_unbiased(
#         source_a=results[0][1, :], source_b=results[0][1, :], target=results[0][1, :], repeats=6
#     )
# )
# correct_trivariate_mi(
#     source_a=results[0][1, :], source_b=results[0][1, :], target=results[0][1, :], repeats=10
# )
# mi_unbiased(source=results[0][1, :], target=results[0][1, :], repeats=6)  # fixed
# entropy(compute_probability_distribution(results[0][1, :]))
# np.log2(np.exp(mutual_info_score(results[0][1, :], results[0][1, :])))

np.float64(0.9761063695264214)

In [9]:
np.sum(
    pid_unbiased(
        target=results_pid[0][0, :],
        source_a=results_pid[0][1, :],
        source_b=results_pid[0][2, :],
        repeats=10,
    )
)

np.float64(0.012109647438087569)

In [7]:
# generate synergistic neurons
# spiking is dependent on cross-correlations


def simulate_spike_trains(
    number_of_neurons,
    timepoints,
    ntrials_per_stimulus,
    lambda1,
    lambda2,
    lambda1_noise,
    lambda2_noise,
    gen_parameter="nonoise",
):

    response_stim_one = np.zeros((number_of_neurons, timepoints, ntrials_per_stimulus))
    response_stim_two = np.zeros((number_of_neurons, timepoints, ntrials_per_stimulus))
    if gen_parameter == "nonoise":

        for neuron in range(number_of_neurons):
            for trials in range(ntrials_per_stimulus):
                response_stim_one[neuron, :, trials] = generate_poission_spikes(
                    np.arange(timepoints), lambda1 / timepoints, 0
                )
                response_stim_two[neuron, :, trials] = generate_poission_spikes(
                    np.arange(timepoints), lambda2 / timepoints, 0
                )
    else:
        for trials in range(ntrials_per_stimulus):
            spikes1_noise = generate_poission_spikes(
                np.arange(timepoints), lambda1_noise / timepoints, 0
            )
            if gen_parameter == "limitnoise":
                spikes2_noise = generate_poission_spikes(
                    np.arange(timepoints), lambda2_noise / timepoints, 0
                )

            for neuron in range(number_of_neurons):
                spikes1 = generate_poission_spikes(np.arange(timepoints), lambda1 / timepoints, 0)
                spikes2 = generate_poission_spikes(np.arange(timepoints), lambda2 / timepoints, 0)
                response_stim_one[neuron, :, trials] = spikes1 + spikes1_noise
                if gen_parameter == "limitnoise":
                    response_stim_two[neuron, :, trials] = spikes2 + spikes2_noise
                else:
                    response_stim_two[neuron, :, trials] = spikes2

    return response_stim_one, response_stim_two


def generate_poission_spikes(time, rate, noise_prob):
    time = np.asarray(time)

    dt = time[1] - time[0]

    if np.isscalar(rate):
        rate = rate * np.ones_like(time)

    spikes = np.zeros_like(time, dtype=int)
    rand_nums = np.random.uniform(0, 1, size=time.shape)
    spikes[rand_nums <= dt * rate] = 1

    if noise_prob != 0:
        rand_nums_noise = np.random.uniform(0, 1, size=time.shape)
        spikes[rand_nums_noise <= noise_prob] = ~spikes[rand_nums_noise <= noise_prob]
    return spikes


nNeurons = 20
nPairs = nNeurons * (nNeurons - 1) / 2
nTrials_per_stim = 200
nShuff = 2
nReps = 10
tPoints = 500


# these parameters are for limiting correlations
lambda_1 = 0.8
lambda_2 = 1.9
lambda_noise_1 = 0.2
lambda_noise_2 = 0.1

stimulus = np.concatenate([np.zeros((nTrials_per_stim)), np.ones((nTrials_per_stim))])

r1_caseone, r2_caseone = simulate_spike_trains(
    number_of_neurons=nNeurons,
    timepoints=tPoints,
    ntrials_per_stimulus=nTrials_per_stim,
    lambda1=lambda_1,
    lambda1_noise=lambda_noise_1,
    lambda2=lambda_2,
    lambda2_noise=lambda_noise_2,
    gen_parameter="nonoise",
)

# these parameters are for enhancing correlations
lambda_1 = 1
lambda_2 = 2
lambda_noise_1 = 1
lambda_noise_2 = 0

r1_casetwo, r2_casetwo = simulate_spike_trains(
    number_of_neurons=nNeurons,
    timepoints=tPoints,
    ntrials_per_stimulus=nTrials_per_stim,
    lambda1=lambda_1,
    lambda1_noise=lambda_noise_1,
    lambda2=lambda_2,
    lambda2_noise=lambda_noise_2,
    gen_parameter="limitnoise",
)

# total spike count for each trial
r1_caseone_over_time = np.sum(r1_caseone, axis=1)
r2_caseone_over_time = np.sum(r2_caseone, axis=1)

r1_casetwo_over_time = np.sum(r1_casetwo, axis=1)
r2_casetwo_over_time = np.sum(r2_casetwo, axis=1)

# just combine

case_one = np.hstack([r1_caseone_over_time, r2_caseone_over_time])
case_two = np.hstack([r1_casetwo_over_time, r2_casetwo_over_time])

In [8]:
case_two.shape, stimulus.shape

((20, 400), (400,))

In [9]:
results = decode_with_sampled_neurons_pid(
    trial_data=stimulus, neural_data=case_two.T, ns_neurons=6
)

In [29]:
results_a = decode_with_sampled_neurons_pid(
    trial_data=stimulus, neural_data=case_one.T, ns_neurons=6
)

In [33]:
pid_results = np.zeros((len(results), 4))
for idx in range(len(results)):
    y, ypreda, ypredb = results[idx]
    pid_results[idx, :] = pid_unbiased(target=y, source_a=ypreda, source_b=ypredb, repeats=10)

In [34]:
pid_results_a = np.zeros((len(results), 4))
for idx in range(len(results_a)):
    y, ypreda, ypredb = results_a[idx]
    pid_results_a[idx, :] = pid_unbiased(target=y, source_a=ypreda, source_b=ypredb, repeats=10)

In [35]:
np.mean(pid_results_a, axis=0)

array([0.0181496 , 0.00106803, 0.42906809, 0.18943806])

In [36]:
np.mean(pid_results, axis=0)

array([ 0.00178944,  0.00301778, -0.00016919,  0.00583965])

In [37]:
from ibl_info.utility import generate_source_ids
from tqdm import tqdm


def compute_pid(data, targets, repeats=5):

    sources = generate_source_ids(data.shape[0])
    pid_information = np.zeros((len(sources), 4))  # neuronsC2 x 4
    for idx in tqdm(
        range(len(sources)), desc="Running for all sources", leave=False
    ):  # this is the place to introduce parallelization
        s1 = sources[idx][0]
        s2 = sources[idx][1]
        X1 = np.asarray(data[s1, :], dtype=np.int32)
        X2 = np.asarray(data[s2, :], dtype=np.int32)
        Y = np.asarray(targets, dtype=np.int32)
        u1, u2, red, syn = pid_unbiased(source_a=X1, source_b=X2, target=Y, repeats=repeats)  # type: ignore
        pid_information[idx, :] = u1, u2, red, syn

    return pid_information

In [38]:
pid_pairwise = compute_pid(data=case_two, targets=stimulus, repeats=10)

In [39]:
pid_pairwise_a = compute_pid(data=case_one, targets=stimulus, repeats=10)

In [45]:
pairwise_case_a = np.mean(pid_pairwise, axis=0)
pairwise_case_b = np.mean(pid_pairwise_a, axis=0)

decoding_case_a = np.mean(pid_results, axis=0)
decoding_case_b = np.mean(pid_results_a, axis=0)

In [47]:
A = pairwise_case_a / np.sum(pairwise_case_a)
B = pairwise_case_b / np.sum(pairwise_case_b)
C = decoding_case_a / np.sum(decoding_case_a)
D = decoding_case_b / np.sum(decoding_case_b)